## Example 1: File type tags from file extensions

In [ ]:
from identify import identify

# tags_from_filename maps a file name to a set of tags without opening the file
test_files = ["script.py", "data.json", "README.md", "config.yaml", "test.js", "style.css", "index.html"]
for filename in test_files:
    print(f"  {filename:>12} -> {sorted(identify.tags_from_filename(filename))}")

assert {"python", "text"} <= identify.tags_from_filename("script.py")
assert {"json", "text"} <= identify.tags_from_filename("data.json")
assert {"markdown", "text"} <= identify.tags_from_filename("README.md")
assert {"yaml", "text"} <= identify.tags_from_filename("config.yaml")
assert "javascript" in identify.tags_from_filename("test.js")
# an unknown extension gives no tags
assert identify.tags_from_filename("archive.unknownext") == set()

## Example 2: Special file names without an extension

In [ ]:
from identify import identify

special_files = {
    "Makefile": "makefile",
    "Dockerfile": "dockerfile",
    ".gitignore": "gitignore",
    "setup.cfg": "ini",
    "requirements.txt": "plain-text",
}
for filename, expected_tag in special_files.items():
    tags = identify.tags_from_filename(filename)
    print(f"  {filename:>16} -> {sorted(tags)}")
    assert expected_tag in tags, (filename, tags)
    assert "text" in tags

## Example 3: Tags from a shebang interpreter

In [ ]:
from identify import identify

for interpreter in ["bash", "sh", "python3.12", "/usr/bin/python3", "node", "no-such-interpreter"]:
    print(f"  {interpreter:>20} -> {sorted(identify.tags_from_interpreter(interpreter))}")

assert "shell" in identify.tags_from_interpreter("bash")
# version suffixes and full paths are reduced to the interpreter name
assert {"python", "python3"} <= identify.tags_from_interpreter("python3.12")
assert "python" in identify.tags_from_interpreter("/usr/bin/python3")
assert "javascript" in identify.tags_from_interpreter("node")
assert identify.tags_from_interpreter("no-such-interpreter") == set()

## Example 4: Tags for real files and directories with tags_from_path

In [ ]:
import os
import stat
import tempfile

from identify import identify

with tempfile.TemporaryDirectory() as temp_dir:
    source = os.path.join(temp_dir, "module.py")
    with open(source, "w") as f:
        f.write("print('hello')\n")

    script = os.path.join(temp_dir, "run")  # no extension: type comes from the shebang
    with open(script, "w") as f:
        f.write("#!/usr/bin/env bash\necho hi\n")
    os.chmod(script, os.stat(script).st_mode | stat.S_IXUSR)

    blob = os.path.join(temp_dir, "blob.bin")
    with open(blob, "wb") as f:
        f.write(bytes(range(256)))

    for path in (source, script, blob, temp_dir):
        print(f"  {os.path.basename(path) or path:>10} -> {sorted(identify.tags_from_path(path))}")

    assert {"file", "text", "python"} <= identify.tags_from_path(source)
    assert {"file", "binary"} <= identify.tags_from_path(blob)
    assert identify.tags_from_path(temp_dir) == {"directory"}
    if os.name == "posix":  # the executable bit and shebang lookup are POSIX concepts
        assert "non-executable" in identify.tags_from_path(source)
        assert {"file", "text", "executable", "shell", "bash"} <= identify.tags_from_path(script)

## Example 5: Text detection and shebang parsing

In [ ]:
import io
import os
import tempfile

from identify import identify

# is_text inspects a byte stream; file_is_text does the same for a path
assert identify.is_text(io.BytesIO(b"plain ascii text\n"))
assert not identify.is_text(io.BytesIO(b"\x00\x01\xff\xfe"))

# parse_shebang returns the interpreter command, with /usr/bin/env removed
assert identify.parse_shebang(io.BytesIO(b"#!/usr/bin/env python3\nprint(1)\n")) == ("python3",)
assert identify.parse_shebang(io.BytesIO(b"no shebang here\n")) == ()

with tempfile.TemporaryDirectory() as temp_dir:
    path = os.path.join(temp_dir, "notes.txt")
    with open(path, "w") as f:
        f.write("hello\n")
    assert identify.file_is_text(path)

# every tag returned above belongs to the published tag vocabulary
for tag in ("python", "shell", "json", "markdown", "directory", "executable", "binary"):
    assert tag in identify.ALL_TAGS, tag
print(f"{len(identify.ALL_TAGS)} known tags")